# SKANN-SSL Pairing Manifest Generator v9

**Project:** SKANN-SSL — Passive Acoustic Vessel Detection & Classification  
**Deployment:** NUWR Goa, seabed hydrophone, 30 m depth, 512 Hz, 28 Jan – 11 Feb 2026  
**Prepared by:** Oravont Systems LLP for Altair Infrasec Pvt Ltd

---

## Design Principle

**Pure temporal self-supervision. No class labels enter the pairing logic.**

Every vessel window is paired exclusively with other windows from the **same event**,
separated by ≥ 90 s. This captures genuine within-transit acoustic variation
(approach → CPA → departure) without injecting any class-level supervision.

Vessel class labels are post-training evaluation metadata only. They appear in the
manifest for UMAP diagnostics but never influence which pairs are formed.

---

## Why this is sufficient

- Event normalisation (event_normalisation.ipynb) produced exactly 50 windows per
  event. With a 30 s window stride, each event spans ≥ 25 min. Every anchor has a
  large temporal pool (≥ 40 eligible partners after gap filter + self-exclusion).
  Short-event fallback is not needed.
- Semantic (cross-event same-class) pairs are excluded by design. They would inject
  class supervision and bias the encoder before training begins.
- Augmentation diversity is already baked into the tensors via event_normalisation.
  No additional augmentation is applied here.

---

## Pair Composition — K=8 per anchor

| Type | Source | Criterion | Slots |
|---|---|---|---|
| `temporal` | Same event | Windows ≥ 90 s apart | 8 |

Partner selection: candidates restricted to top `HARD_FRAC` (25%) most dissimilar
in IQR-scaled 6D feature space, then K most dissimilar selected. This ensures
partners span the acoustic range of the transit.

---

## Inputs

| File | Description |
|---|---|
| `window_pool_50.csv` | 7,050 vessel windows (141 events × 50), with event_id, offset_sec, 6D features |
| `havs_events_with_sessions.csv` | HAVS events with vessel_class — evaluation metadata only |

## Output

`pairing_manifest.csv` — schema:  
`anchor, partner, pair_type, split, feat_cosine_dist, rms_ratio, gap_sec, vessel_class`

- `vessel_class` : metadata for post-training UMAP diagnostics only
- `pair_type`    : always `temporal` in this version

## Next step

Feed `pairing_manifest.csv` into `SKANN_SSL_Training.ipynb`.


## Cell 1 — Imports + Mount Drive

In [ ]:
import os
import ast
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


## Cell 2 — Configuration

In [ ]:
# ── Input paths ────────────────────────────────────────────────────────────
BASE_DIR             = '/content/drive/MyDrive/SKANN_SSL'
VESSEL_FEATURES_PATH = os.path.join(BASE_DIR, 'ssl_data_50w',         'window_pool_50.csv')
TENSOR_VESSEL_DIR    = os.path.join(BASE_DIR, 'v5_data', 'tensors', 'vessel')

# ── Output paths ───────────────────────────────────────────────────────────
OUT_DIR              = os.path.join(BASE_DIR, 'ssl_data_50w')
PAIRING_MANIFEST_OUT = os.path.join(OUT_DIR,  'pairing_manifest.csv')
CLASS_METADATA_OUT   = os.path.join(OUT_DIR,  'window_class_metadata.csv')

os.makedirs(OUT_DIR, exist_ok=True)

# ── Pairing parameters ─────────────────────────────────────────────────────
K            = 8       # temporal partners per anchor
MIN_GAP_SEC  = 90.0    # minimum temporal gap within event (seconds)
HARD_FRAC    = 0.25    # restrict candidates to top 25% most dissimilar in feature space
RANDOM_SEED  = 42

# ── 6D feature columns ─────────────────────────────────────────────────────
FEAT_COLS = ['logbp_1_5', 'logbp_5_15', 'logbp_15_40',
             'logbp_40_90', 'logbp_90_180', 'peakiness']

print('Configuration:')
print(f'  K (partners per anchor) : {K}')
print(f'  MIN_GAP_SEC             : {MIN_GAP_SEC}s')
print(f'  HARD_FRAC               : top {int(HARD_FRAC*100)}% most dissimilar')
print(f'  Pair type               : temporal only (same event, >=90s gap)')
print(f'  Input manifest          : {VESSEL_FEATURES_PATH}')
print(f'  Tensor directory        : {TENSOR_VESSEL_DIR}')
print(f'  Output manifest         : {PAIRING_MANIFEST_OUT}')


Configuration:
  K (partners per anchor) : 8
  MIN_GAP_SEC             : 90.0s
  HARD_FRAC               : top 25% most dissimilar
  Pair type               : temporal only (same event, >=90s gap)
  Input manifest          : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_pool_50.csv
  Tensor directory        : /content/drive/MyDrive/SKANN_SSL/v5_data/tensors/vessel
  Output manifest         : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/pairing_manifest.csv


## Cell 3 — Load Features + HAVS Metadata

In [ ]:
# ── File guard ────────────────────────────────────────────────────────────
if not os.path.exists(VESSEL_FEATURES_PATH):
    raise FileNotFoundError(
        f'Missing: {VESSEL_FEATURES_PATH}\n'
        f'  Run event_normalisation_v4.ipynb first.'
    )

# ── Load vessel windows ────────────────────────────────────────────────────
vdf = pd.read_csv(VESSEL_FEATURES_PATH)

required_cols = FEAT_COLS + ['pre_norm_rms', 'offset_sec', 'event_id', 'split',
                              'tensor_file', 'assessment']
for col in required_cols:
    assert col in vdf.columns, f'window_pool_50.csv missing column: {col}'

null_check_cols = FEAT_COLS + ['pre_norm_rms', 'offset_sec', 'event_id', 'split', 'tensor_file']
nulls = vdf[null_check_cols].isnull().sum()
assert nulls.sum() == 0, f'Nulls in pairing-critical columns:\n{nulls[nulls>0]}'

wpev = vdf.groupby('event_id').size()
assert (wpev == 50).all(), (
    f'Not all events have exactly 50 windows.\n'
    f'Offending events:\n{wpev[wpev != 50]}'
)

# ── Verify tensors exist on disk ───────────────────────────────────────────
missing = [f for f in vdf['tensor_file']
           if not os.path.exists(os.path.join(TENSOR_VESSEL_DIR, f))]
if missing:
    raise FileNotFoundError(
        f'{len(missing)} tensor files missing from {TENSOR_VESSEL_DIR}\n'
        f'First missing: {missing[0]}'
    )

# ── Derive vessel_class from assessment column (already in window_pool_50.csv) ─
# assessment values: 'CONFIRMED VESSEL', 'PROBABLE VESSEL', etc.
# vessel_class is for post-training UMAP diagnostics ONLY — never gates pairing.
vdf['vessel_class'] = vdf['assessment'].fillna('unclassified')

print('=== Vessel Window Pool (50w) ===')
print(f'  Total windows  : {len(vdf):,}')
print(f'  Events         : {vdf.event_id.nunique()}')
print(f'  Windows/event  : all = {wpev.min()} ✓')
print(f'  Split          : train={(vdf.split=="train").sum():,}  val={(vdf.split=="val").sum():,}')
print(f'  Tensors on disk: {len(vdf):,} / {len(vdf):,} ✓')
print(f'\n  Assessment distribution:')
print(vdf['assessment'].value_counts().to_string())


=== Vessel Window Pool (50w) ===
  Total windows  : 7,050
  Events         : 141
  Windows/event  : all = 50 ✓
  Split          : train=5,800  val=1,250
  Tensors on disk: 7,050 / 7,050 ✓

  Assessment distribution:
assessment
PROBABLE VESSEL     3650
CONFIRMED VESSEL    3400


## Cell 4 — Helper Functions

In [ ]:
def iqr_scale(X):
    """Robust IQR scaling across all rows. Stable against impulsive noise."""
    q25 = np.percentile(X, 25, axis=0)
    q75 = np.percentile(X, 75, axis=0)
    iqr = np.where((q75 - q25) < 1e-8, 1e-8, q75 - q25)
    return (X - np.median(X, axis=0)) / iqr


def cosine_distances(anchor_feat, cand_feats):
    """Cosine distance from anchor to each row of cand_feats."""
    a     = anchor_feat / max(np.linalg.norm(anchor_feat), 1e-12)
    norms = np.linalg.norm(cand_feats, axis=1, keepdims=True)
    norms = np.where(norms < 1e-12, 1e-12, norms)
    return 1.0 - (cand_feats / norms) @ a


def select_partners(anchor_feat, cand_df, feat_map, n_slots):
    """
    Select n_slots partners from cand_df using hard-negative mining.

    Strategy:
      1. Compute cosine distance from anchor to every candidate in 6D feature space.
      2. Restrict to top HARD_FRAC most dissimilar (avoids trivially similar pairs
         while not selecting maximally different outliers).
      3. Return the n_slots most dissimilar within that pool.

    With 50 windows per event and MIN_GAP_SEC=90s, cand_df will always have
    >= 40 candidates — n_slots=8 is always satisfiable.
    """
    cand_feats = np.array([feat_map[tf] for tf in cand_df['tensor_file']])
    dists      = cosine_distances(anchor_feat, cand_feats)

    threshold = np.percentile(dists, (1.0 - HARD_FRAC) * 100)
    mask      = dists >= threshold
    if mask.sum() < n_slots:
        mask = np.ones(len(dists), dtype=bool)   # fallback: use all

    dists_f   = dists[mask]
    cand_f    = cand_df.iloc[np.where(mask)[0]]
    order     = np.argsort(-dists_f)
    take      = min(n_slots, len(order))
    return cand_f.iloc[order[:take]], dists_f[order[:take]]


def make_row(anchor_tf, anchor_rms, anchor_off,
             partner_tf, partner_rms, partner_off, dist, split):
    gap       = round(abs(float(partner_off) - float(anchor_off)), 1)
    rms_ratio = max(anchor_rms, partner_rms) / max(min(anchor_rms, partner_rms), 1e-12)
    return {
        'anchor'          : anchor_tf,
        'partner'         : partner_tf,
        'pair_type'       : 'temporal',
        'split'           : split,
        'feat_cosine_dist': round(float(dist), 6),
        'rms_ratio'       : round(float(rms_ratio), 4),
        'gap_sec'         : gap,
    }


print('Helper functions defined.')
print(f'  select_partners: HARD_FRAC={HARD_FRAC}, K={K}, MIN_GAP_SEC={MIN_GAP_SEC}s')

Helper functions defined.
  select_partners: HARD_FRAC=0.25, K=8, MIN_GAP_SEC=90.0s


## Cell 5 — Build Pairing Manifest

In [ ]:
# ── Global IQR feature scaling ────────────────────────────────────────────
X_raw    = vdf[FEAT_COLS].values.astype(float)
X_sc     = iqr_scale(X_raw)
feat_map = {tf: X_sc[i] for i, tf in enumerate(vdf['tensor_file'].values)}

rng      = np.random.default_rng(seed=RANDOM_SEED)
all_rows = []

print(f'Building pairs: {len(vdf):,} anchors × {K} partners each')
print(f'Pair type: temporal only (same event, >= {MIN_GAP_SEC:.0f}s gap)\n')

for split in ['train', 'val']:
    sv           = vdf[vdf['split'] == split].copy().reset_index(drop=True)
    event_groups = {eid: grp for eid, grp in sv.groupby('event_id')}

    for _, anchor in sv.iterrows():
        a_tf    = anchor['tensor_file']
        a_feat  = feat_map[a_tf]
        a_rms   = float(anchor['pre_norm_rms'])
        a_off   = float(anchor['offset_sec'])
        a_event = anchor['event_id']
        # Temporal pool: same event, not self, >= MIN_GAP_SEC away
        ev_grp = event_groups.get(a_event, sv.iloc[0:0])
        pool   = ev_grp[
            (ev_grp['tensor_file'] != a_tf) &
            (np.abs(ev_grp['offset_sec'].astype(float) - a_off) >= MIN_GAP_SEC)
        ]

        if len(pool) == 0:
            # Should not happen with 50 windows/event — log if it does
            print(f'  WARNING: no temporal pool for {a_tf} (event {a_event}, split {split})')
            continue

        sel, dists = select_partners(a_feat, pool, feat_map, K)

        for (_, row), dist in zip(sel.iterrows(), dists):
            all_rows.append(make_row(
                a_tf, a_rms, a_off,
                row['tensor_file'], float(row['pre_norm_rms']),
                float(row['offset_sec']), dist, split
            ))

print(f'Pairs generated : {len(all_rows):,}')
print(f'Expected        : {len(vdf) * K:,}  ({len(vdf):,} anchors × {K})')
if len(all_rows) < len(vdf) * K:
    shortfall = len(vdf) * K - len(all_rows)
    print(f'WARNING: {shortfall} fewer pairs than expected — check WARNING lines above')

Building pairs: 7,050 anchors × 8 partners each
Pair type: temporal only (same event, >= 90s gap)

Pairs generated : 56,400
Expected        : 56,400  (7,050 anchors × 8)


## Cell 6 — Validate, Statistics, Save

In [ ]:
manifest = pd.DataFrame(all_rows)
print(f'Total rows : {len(manifest):,}\n')

# ── Integrity checks ───────────────────────────────────────────────────────
assert (manifest['anchor'] == manifest['partner']).sum() == 0,\
    'FAIL: self-pairs found'
print('Self-pair check              : PASSED')

assert manifest[['anchor','partner','pair_type','split']].isnull().sum().sum() == 0,\
    'FAIL: nulls in key columns'
print('Null check                   : PASSED')

known = set(vdf['tensor_file'])
unknown = set(manifest['partner']) - known
assert len(unknown) == 0, f'FAIL: {len(unknown)} unknown partner filenames'
print('Filename integrity           : PASSED')

dupes = manifest.groupby(['anchor', 'partner', 'split']).size()
assert dupes.max() == 1, 'FAIL: duplicate (anchor, partner) pairs found'
print('Duplicate pair check         : PASSED')

train_anchors = set(manifest[manifest['split'] == 'train']['anchor'])
val_anchors   = set(manifest[manifest['split'] == 'val'  ]['anchor'])
assert len(train_anchors & val_anchors) == 0,\
    'FAIL: anchors appear in both train and val'
print('Train/val isolation          : PASSED')

assert (manifest['pair_type'] == 'temporal').all(),\
    'FAIL: unexpected pair types found'
print('Pair type purity             : PASSED (temporal only)')

gap_violations = (manifest['gap_sec'] < MIN_GAP_SEC).sum()
assert gap_violations == 0, f'FAIL: {gap_violations} pairs below MIN_GAP_SEC={MIN_GAP_SEC}s'
print(f'Gap constraint               : PASSED (all gaps >= {MIN_GAP_SEC:.0f}s)')

# ── Statistics ─────────────────────────────────────────────────────────────
print('\n=== MANIFEST STATISTICS ===')
print(f'  Total pairs   : {len(manifest):,}')
print(f'  Train pairs   : {(manifest.split=="train").sum():,}')
print(f'  Val pairs     : {(manifest.split=="val").sum():,}')

ppa = manifest.groupby('anchor').size()
print(f'\n  Pairs per anchor:')
print(f'    min={ppa.min()}  median={ppa.median():.0f}  max={ppa.max()}  mean={ppa.mean():.1f}')
for n in sorted(ppa.unique()):
    c = (ppa == n).sum()
    tag = '  <-- K target' if n == K else ''
    print(f'    {n} pairs : {c:,} anchors{tag}')

d = manifest['feat_cosine_dist']
print(f'\n  Feat cosine distance:')
print(f'    mean={d.mean():.4f}  std={d.std():.4f}  p10={np.percentile(d,10):.4f}  p90={np.percentile(d,90):.4f}')

gaps = manifest['gap_sec']
print(f'\n  Temporal gap (sec):')
print(f'    min={gaps.min():.0f}  median={gaps.median():.0f}  max={gaps.max():.0f}')

# ── Save ───────────────────────────────────────────────────────────────────
col_order = ['anchor', 'partner', 'pair_type', 'split',
             'feat_cosine_dist', 'rms_ratio', 'gap_sec']
manifest[col_order].to_csv(PAIRING_MANIFEST_OUT, index=False)
sz = os.path.getsize(PAIRING_MANIFEST_OUT) / 1024
print(f'\nSaved : {PAIRING_MANIFEST_OUT}')
print(f'Rows  : {len(manifest):,}  |  Size: {sz:.1f} KB')

# ── Save class metadata (evaluation use only — never read by training loop) ──
class_meta = vdf[['tensor_file', 'event_id', 'vessel_class']].copy()
class_meta.to_csv(CLASS_METADATA_OUT, index=False)
sz_meta = os.path.getsize(CLASS_METADATA_OUT) / 1024
print(f'Saved : {CLASS_METADATA_OUT}')
print(f'Rows  : {len(class_meta):,}  |  Size: {sz_meta:.1f} KB')
print('  NOTE: window_class_metadata.csv is for post-training evaluation only.')
print('        The training DataLoader must never open this file.\n')

print('=== DATALOADER CONTRACT ===')
print(f'  Train pairs      : {(manifest.split=="train").sum():,}')
print(f'  Val pairs        : {(manifest.split=="val").sum():,}')
print( '  Schema           : anchor, partner, pair_type, split,')
print( '                     feat_cosine_dist, rms_ratio, gap_sec, vessel_class')
print( '  Augmentation     : NONE in DataLoader — tensors loaded as-is')
print( '  vessel_class     : NOT in manifest — see window_class_metadata.csv')

print('\n=== POST-TRAINING DIAGNOSTICS ===')
print('  1. UMAP coloured by vessel_class — did embeddings separate vessel types?')
print('  2. UMAP coloured by event_id     — should NOT cluster by event (collapse check)')
print('  3. corr(embedding, blade_rate_hz) — is the embedding physically interpretable?')
print('  4. corr(embedding, shaft_rate_hz) — is the embedding physically interpretable?')

print('\n=== NEXT STEP ===')
print('  SKANN_SSL_Training.ipynb')
print(f'  PAIRING_MANIFEST_PATH = \'{PAIRING_MANIFEST_OUT}\'')
print(f'  TENSOR_VESSEL_DIR     = \'{TENSOR_VESSEL_DIR}\'')

Total rows : 56,400

Self-pair check              : PASSED
Null check                   : PASSED
Filename integrity           : PASSED
Duplicate pair check         : PASSED
Train/val isolation          : PASSED
Pair type purity             : PASSED (temporal only)
Gap constraint               : PASSED (all gaps >= 90s)

=== MANIFEST STATISTICS ===
  Total pairs   : 56,400
  Train pairs   : 46,400
  Val pairs     : 10,000

  Pairs per anchor:
    min=8  median=8  max=8  mean=8.0
    8 pairs : 7,050 anchors  <-- K target

  Feat cosine distance:
    mean=1.2487  std=0.6119  p10=0.1926  p90=1.8499

  Temporal gap (sec):
    min=90  median=180  max=4650

Saved : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/pairing_manifest.csv
Rows  : 56,400  |  Size: 5037.5 KB
Saved : /content/drive/MyDrive/SKANN_SSL/ssl_data_50w/window_class_metadata.csv
Rows  : 7,050  |  Size: 322.8 KB
  NOTE: window_class_metadata.csv is for post-training evaluation only.
        The training DataLoader must never ope